# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** In Silico Computational Biology & Protein Topology

---
*Note: This notebook illustrates the direct application of object-oriented vectors in data science onto the three-dimensional structure of a macromolecule. By transforming static atomic coordinates into a matrix of relative distances, the seasonal behavior of protein density is isolated—a mathematical abstraction equivalent to mapping biological active sites.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("mako")

### 1. Direct Atomic Coordinate Ingestion (.pdb)
We will focus our *in silico* matrix analysis by exclusively isolating Alpha-Carbon (CA) atoms, which geometrically represent the functional vertices of the amino acid backbone in the protein (1UBQ: Ubiquitin).

In [ ]:
def parse_pdb_coordinates(filepath):
    """
    Extracts 3D coordinates specifically for Alpha-Carbons (CA)
    from a raw Protein Data Bank file, disregarding non-structural metadata.
    """
    ca_atoms = []
    with open(filepath, 'r') as file:
        for line in file:
            # Coordinate records in PDB format always start with ATOM
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                residue_num = int(line[22:26].strip())
                residue_name = line[17:20].strip()
                
                # Orthogonal coordinates for X, Y, Z in Angstroms
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                
                ca_atoms.append({
                    'Residue_Num': residue_num,
                    'Residue_Name': residue_name,
                    'X': x, 'Y': y, 'Z': z
                })
                
    return pd.DataFrame(ca_atoms)

df_ca = parse_pdb_coordinates('../data/1ubq.pdb')
print(f"[*] Extracted {len(df_ca)} structural nodes (Alpha-Carbon Atoms).")
df_ca.head()

### 2. Spatial Transformation: Euclidean Distance Matrix
By permuting Cartesian positions against themselves to produce an $N 	imes N$ symmetric matrix, we convert the three-dimensional protein into a 2D graph of atomic closeness, identifying functionally dense zones.

In [ ]:
coords = df_ca[['X', 'Y', 'Z']].values

# Vectorized euclidean distance matrix calculation
dist_mat = distance_matrix(coords, coords)

# Create Heatmap for visual clustering
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    dist_mat, 
    cmap="viridis_r", # Reversed so darker equals closer (high density)
    square=True, 
    cbar_kws={'label': 'Geometric Distance (Angstroms)'},
    ax=ax
)

ax.set_title("Protein Topology (1UBQ): Side Chain Density", fontsize=14, pad=15)
ax.set_xlabel("Amino Acid Residue Index")
ax.set_ylabel("Amino Acid Residue Index")

plt.show()

### Topological Conclusion
What in the biological plane represented the complex globular folding of ubiquitin, in the heat map translates into high-intensity anomalies off its main diagonal. These bands (Cartesian distances near $0Å$) algorithmically map the compression of distant regions of the primary chain that bind spatially to form possible **hydrogen bonds** and the functional hydrophobic core.